#Init

In [0]:
from pyspark.sql.functions import col, to_date, expr, round as spark_round

#Reading from bronze table

In [0]:
df = spark.read.table("workspace.bronze.gcp_billing_raw")

#Data Transformations

In [0]:
#Transformations
silver_df = df.withColumn("usage_date", to_date(col("usage_start_time"))) \
              .withColumn("total_credits", expr("coalesce(aggregate(credits.amount, CAST(0.0 AS DOUBLE), (acc, x) -> acc + x), CAST(0.0 AS DOUBLE))")) \
                .withColumn("net_cost", spark_round(col("cost")+col("total_credits"), 4))

#verify
display(silver_df.select("usage_date","cost","total_credits","net_cost"))

#Writing Data into Silver Layer

In [0]:
silver_df.write.mode("overwrite").saveAsTable("workspace.silver.gcp_billing_clean")
print("Data successfully landed in the silver layer")